In [ ]:
import optuna
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import mutual_info_regression

# 1. Paths & Load Data
target_name = 'SalePrice'
train_url = '../input/train.csv'
test_url = '../input/test.csv'

X_full = pd.read_csv(train_url, index_col='Id')
X_test_full = pd.read_csv(test_url, index_col='Id')

X_full.head(20)

In [ ]:
# Remove rows with missing target
X_full.dropna(axis=0, subset=[target_name], inplace=True)
y = X_full[target_name]
X_full.drop([target_name], axis=1, inplace=True)

X_full.head(20)

In [ ]:
missing_val_count_by_column = X_full.isnull().sum()
missing_cols = missing_val_count_by_column[missing_val_count_by_column > 0]

missing_df = pd.DataFrame({
    'Missing Values': missing_cols,
    'Percentage (%)': (missing_cols / len(X_full)) * 100
}).sort_values(by='Missing Values', ascending=False)

print("Missing values table for training data")
print("-"*40)
print(missing_df)


missing_val_count_by_column = X_test_full.isnull().sum()
missing_cols = missing_val_count_by_column[missing_val_count_by_column > 0]

print("="*40)

missing_df = pd.DataFrame({
    'Missing Values': missing_cols,
    'Percentage (%)': (missing_cols / len(X_test_full)) * 100
}).sort_values(by='Missing Values', ascending=False)

print("Missing values table for test data")
print("-"*40)
print(missing_df)

In [ ]:
def mathematical_transforms(df):
    X = pd.DataFrame(index=df.index)
    X['LivLotRatio'] = df['GrLivArea'] / (df['LotArea'] + 1)
    X['Spaciousness'] = (df['1stFlrSF'] + df['2ndFlrSF']) / (df['TotRmsAbvGrd'] + 1)
    X['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    return X

def counts(df):
    X = pd.DataFrame(index=df.index)
    porch_cols = ["WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]
    X['PorchTypes'] = (df[porch_cols] > 0.0).sum(axis=1)
    X['TotalBaths'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    return X

def house_age(df):
    X = pd.DataFrame(index=df.index)
    X['HouseAge'] = df['YrSold'] - df['YearBuilt']
    X['RemodelAge'] = df['YrSold'] - df['YearRemodAdd']
    return X

def quality_condition_interactions(df):
    X = pd.DataFrame(index=df.index)
    
    X['Qual_x_Cond'] = df['OverallQual'] * df['OverallCond']
    
    qual_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    
    bsmt_q = df['BsmtQual'].fillna('None').map(qual_map).astype(float)
    bsmt_c = df['BsmtCond'].fillna('None').map(qual_map).astype(float)
    X['Bsmt_Qual_x_Cond'] = bsmt_q * bsmt_c
    
    gar_q = df['GarageQual'].fillna('None').map(qual_map).astype(float)
    gar_c = df['GarageCond'].fillna('None').map(qual_map).astype(float)
    X['Garage_Qual_x_Cond'] = gar_q * gar_c
    
    return X

def sqrt_area_transforms(df):
    X = pd.DataFrame(index=df.index)
    area_cols = ['GrLivArea', 'TotalBsmtSF', 'LotArea', '1stFlrSF', '2ndFlrSF', 'GarageArea']
    
    for col in area_cols:
        X[f'Sqrt_{col}'] = np.sqrt(df[col].clip(lower=0))
        
    return X

def log_transforms(df):
    X = pd.DataFrame(index=df.index)
    skewed_cols = ['LotArea', 'LotFrontage', 'MasVnrArea', 'OpenPorchSF', 'WoodDeckSF']
    
    for col in skewed_cols:
        val = df[col].fillna(0).clip(lower=0)
        X[f'Log_{col}'] = np.log1p(val)
        
    return X

def numeric_categorical_interactions(df):
    X = pd.DataFrame(index=df.index)
    qual_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    
    bsmt_qual_num = df['BsmtQual'].fillna('None').map(qual_map).astype(float)
    X['Bsmt_Area_x_Qual'] = df['TotalBsmtSF'].fillna(0) * bsmt_qual_num
    
    garage_qual_num = df['GarageQual'].fillna('None').map(qual_map).astype(float)
    X['Garage_Area_x_Qual'] = df['GarageArea'].fillna(0) * garage_qual_num
    
    return X


def neighborhood_advanced_stats(df):
    X = pd.DataFrame(index=df.index)
    nhbd_stats = X_full.groupby('Neighborhood')['GrLivArea'].agg(['mean', 'median', 'std']).fillna(0)
    
    med = df['Neighborhood'].map(nhbd_stats['median']).fillna(df['GrLivArea'].median())
    mean = df['Neighborhood'].map(nhbd_stats['mean']).fillna(df['GrLivArea'].mean())
    std = df['Neighborhood'].map(nhbd_stats['std']).replace(0, 1).fillna(1)
    
    X['Diff_From_Nhbd_Median'] = df['GrLivArea'] - med
    
    X['ZScore_GrLivArea_Nhbd'] = (df['GrLivArea'] - mean) / std
    
    return X

In [ ]:
cluster_features = [
    "LotArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "GrLivArea",
]

scaler = StandardScaler()
X_full_scaled = scaler.fit_transform(X_full[cluster_features].fillna(0))

kmeans = KMeans(n_clusters=10, n_init=20, random_state=0)
kmeans.fit(X_full_scaled)

def add_cluster_features(df):
    X = pd.DataFrame(index=df.index)
    
    df_scaled = scaler.transform(df[cluster_features].fillna(0))
    
    X['Cluster'] = kmeans.predict(df_scaled).astype(str)
    
    distances = kmeans.transform(df_scaled)
    for i in range(distances.shape[1]):
        X[f'Centroid_Dist_{i}'] = distances[:, i]
        
    return X

In [ ]:
pca_features = [
    "GarageArea",
    "YearRemodAdd",
    "TotalBsmtSF",
    "GrLivArea",
]

pca_scaler = StandardScaler()
X_full_pca_scaled = pca_scaler.fit_transform(X_full[pca_features].fillna(0))

pca = PCA(n_components=4, random_state=0)
pca.fit(X_full_pca_scaled)

def add_pca_components(df):
    X = pd.DataFrame(index=df.index)
    df_pca_scaled = pca_scaler.transform(df[pca_features].fillna(0))
    components = pca.transform(df_pca_scaled)
    
    for i in range(components.shape[1]):
        X[f'PC_{i+1}'] = components[:, i]
        
    return X

def pca_inspired_features(df):
    X = pd.DataFrame(index=df.index)
    X['Remod_x_BsmtArea'] = df['YearRemodAdd'] * df['TotalBsmtSF']
    return X

def indicate_outliers(df):
    X = pd.DataFrame(index=df.index)
    is_edwards = (df['Neighborhood'] == "Edwards")
    is_partial = (df['SaleCondition'] == "Partial")
    X['Is_Outlier_Case'] = (is_edwards & is_partial).astype(int)
    return X

In [ ]:
def create_features(df):
    X = df.copy()
    
    X = X.join(mathematical_transforms(X))
    X = X.join(counts(X))
    X = X.join(house_age(X))
    X = X.join(quality_condition_interactions(X))
    X = X.join(sqrt_area_transforms(X))
    X = X.join(log_transforms(X))
    X = X.join(numeric_categorical_interactions(X))
    X = X.join(neighborhood_advanced_stats(X))
    X = X.join(add_cluster_features(X))
    X = X.join(add_pca_components(X))
    X = X.join(pca_inspired_features(X))
    X = X.join(indicate_outliers(X))

    return X

X = create_features(X_full)
X_test = create_features(X_test_full)

print("Feature engineering completed successfully")

In [ ]:
def plot_variance(pca, width=8, dpi=100):
    fig, axs = plt.subplots(1, 2)
    n = pca.n_components_
    grid = np.arange(1, n + 1)
    
    evr = pca.explained_variance_ratio_
    axs[0].bar(grid, evr, color='skyblue', edgecolor='black')
    axs[0].set(
        xlabel="Component", title="% Explained Variance", ylim=(0.0, 1.0)
    )
    axs[0].set_xticks(grid)
    axs[0].grid(axis='y', linestyle='--', alpha=0.7)

    cv = np.cumsum(evr)
    axs[1].plot(np.r_[0, grid], np.r_[0, cv], "o-", color='crimson')
    axs[1].set(
        xlabel="Component", title="Cumulative Variance", ylim=(0.0, 1.0)
    )
    axs[1].set_xticks(np.r_[0, grid])
    axs[1].grid(linestyle='--', alpha=0.7)

    fig.set(figwidth=width, dpi=dpi)
    plt.tight_layout()
    plt.show()

print("--- The variance ratio explained by PCA ---")
plot_variance(pca)


In [ ]:
def corrplot(df, method="pearson", annot=True, **kwargs):
    corr = df.corr(method=method, numeric_only=True)
    
    g = sns.clustermap(
        corr,
        vmin=-1.0,
        vmax=1.0,
        cmap="icefire",
        method="complete",
        annot=annot,
        **kwargs,
    )
    plt.title("Clustered Correlation Matrix", pad=15)
    plt.show()

print("--- Link map of PCA features with price ---")
pca_analysis_df = X_full[pca_features].copy()
pca_analysis_df['SalePrice'] = y

corrplot(pca_analysis_df, annot=True, figsize=(8, 8))

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
    index=pca_features
)

print("\n--- PCA Loadings Table (Loadings) ---")
print(loadings.round(3))

In [ ]:
# 2. Select Columns
numerical_cols = [cname for cname in X.columns if X[cname].dtype in ['int64', 'float64']]
categorical_cols = [cname for cname in X.columns if X[cname].nunique() < 10 and X[cname].dtype == "object"]

my_cols = categorical_cols + numerical_cols
X = X[my_cols].copy()
X_test = X_test[my_cols].copy()

X.head(20)

In [ ]:
# 3. Pipeline Setup
numerical_transformer = SimpleImputer(strategy='mean', add_indicator=True)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]
)

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

print("Preprocessing completed successfully")

In [ ]:
# 5. Calculating and graphing Mutual Information (MI)
print('--- Calculating & Plotting Mutual Information ---')

X_transformed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

mi_scores = mutual_info_regression(X_transformed, y, random_state=0)
mi_series = pd.Series(mi_scores, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, round(mi_series.count() * 0.3)), dpi=100)
mi_series.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Mutual Information Scores', fontsize=12, pad=15)
plt.xlabel('MI Score', fontsize=10)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("Top Feature Scores:")
print(mi_series.sort_values(ascending=False))

In [ ]:
informative_features = mi_series[mi_series > 0.0].index.tolist()

print(f"Total number of features before deletion: {len(mi_series)}")
print(f"Number of features retained (greater than 0): {len(informative_features)}")
print(f"Number of features removed: {len(mi_series) - len(informative_features)}")

X_transformed = preprocessor.fit_transform(X)
X_test_transformed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

X_filtered = pd.DataFrame(X_transformed, columns=feature_names, index=X.index)[informative_features]
X_test_filtered = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)[informative_features]

X_filtered.head(20)

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0),
        'random_state': 0,
        'n_jobs': -1
    }

    model = XGBRegressor(**params)

    scores = -1 * cross_val_score(
        model, X_filtered, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
    )

    return scores.mean()

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction='minimize')

print("The search is on for the best parameters...")
study.optimize(objective, n_trials=40)

print("\n" + "="*40)
print(f"🔥 Best CV MAE: ${study.best_value:,.2f}")
print("🎯 Best Parameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")
print("="*40)

In [ ]:
best_model = XGBRegressor(**study.best_params, random_state=0, n_jobs=-1)

best_model.fit(X_filtered, y)

test_preds = best_model.predict(X_test_filtered)

print("Model fit completed successfully")

In [ ]:
id_cols = [col for col in X_test_full.columns if col.lower() == 'id']

if id_cols:
    test_id = X_test_full[id_cols[0]]
else:
    test_id = X_test_full.index


output = pd.DataFrame({
    'Id': test_id, 
    target_name: test_preds
})

output.to_csv('submission.csv', index=False)

print("Your submission was successfully saved!")